In [ ]:
%pip install -q torch pandas scikit-learn tqdm

In [ ]:
import math

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def make_padding_mask(seq, pad_idx=0):
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


def make_causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len)).bool()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T]


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    output = weights @ V

    return output, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, _ = x.shape
        x = x.view(B, T, self.h, self.d_k)
        return x.transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        out, _ = scaled_dot_product_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.W_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x


class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causal_mask=None, pad_mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, causal_mask)))

        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, pad_mask)))

        x = self.norm3(x + self.dropout(self.ffn(x)))

        return x


class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, enc_out, causal_mask=None, pad_mask=None):
        for layer in self.layers:
            x = layer(x, enc_out, causal_mask, pad_mask)
        return x

In [ ]:
class ChatbotTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_layers=3, num_heads=8,
                 d_ff=512, max_len=64, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx

        self.embed_layer = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pe_layer = PositionalEncoding(d_model, max_len=max_len)
        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff, dropout)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff, dropout)

        self.output_linear = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        src_pad_mask = make_padding_mask(src, self.pad_idx)
        x = self.dropout(self.pe_layer(self.embed_layer(src)))
        enc_out = self.encoder(x, src_pad_mask)
        return enc_out, src_pad_mask

    def decode(self, tgt, enc_out, src_pad_mask):
        T = tgt.size(1)
        causal_mask = make_causal_mask(T).to(tgt.device)
        tgt_pad_mask = make_padding_mask(tgt, self.pad_idx)
        combined_mask = causal_mask & tgt_pad_mask

        x = self.dropout(self.pe_layer(self.embed_layer(tgt)))
        dec_out = self.decoder(x, enc_out, combined_mask, src_pad_mask)

        return self.output_linear(dec_out)

    def forward(self, src, tgt):
        enc_out, src_pad_mask = self.encode(src)
        return self.decode(tgt, enc_out, src_pad_mask)

In [ ]:
from collections import Counter

SPECIAL_TOKENS = ("<pad>", "<sos>", "<eos>", "<unk>")


def create_vocab(sentences, special_tokens=SPECIAL_TOKENS, min_freq=2):
    # min_freq
    counter = Counter(w for s in sentences for w in s.split())
    words = sorted(w for w, c in counter.items() if c >= min_freq)
    vocab = list(special_tokens) + words
    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for i, w in enumerate(vocab)}
    return vocab, stoi, itos


def encode(sentence, stoi):
    ids = [stoi.get(w, stoi["<unk>"]) for w in sentence.split()]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]


def decode(ids, itos):
    specials = {"<pad>", "<sos>", "<eos>"}
    words = []
    for i in ids:
        w = itos.get(int(i), "<unk>")
        if w == "<eos>":
            break
        if w in specials:
            continue
        words.append(w)
    return " ".join(words)


def pad_ids(ids, max_len, pad_idx=0):
    ids = ids[:max_len]
    return ids + [pad_idx] * (max_len - len(ids))


class ChatbotDataset(Dataset):
    def __init__(self, questions, answers, stoi, max_len=30):
        self.questions = questions
        self.answers = answers
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        src_ids = pad_ids(encode(self.questions[idx], self.stoi), self.max_len)

        tgt_full = pad_ids(encode(self.answers[idx], self.stoi), self.max_len + 1)
        tgt_in = tgt_full[:-1]
        tgt_out = tgt_full[1:]

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_in, dtype=torch.long),
            torch.tensor(tgt_out, dtype=torch.long),
        )


def generate_reply(model, question, stoi, itos, max_len=30, device="cpu"):
    model.eval()
    src_ids = pad_ids(encode(question, stoi), max_len)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    with torch.no_grad():
        enc_out, src_pad_mask = model.encode(src)

        tgt_ids = [stoi["<sos>"]]
        for _ in range(max_len):
            tgt = torch.tensor([tgt_ids], dtype=torch.long, device=device)
            logits = model.decode(tgt, enc_out, src_pad_mask)
            next_id = logits[0, -1].argmax(-1).item()

            if next_id == stoi["<eos>"]:
                break
            tgt_ids.append(next_id)

    reply = decode(tgt_ids[1:], itos)
    return reply if reply else "음... 잘 모르겠어요."

In [ ]:
data_url = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
df = pd.read_csv(data_url)

df = df.dropna(subset=["Q", "A"])
print(df.shape)
df.head()

In [ ]:
questions, answers, labels = df["Q"].tolist(), df["A"].tolist(), df["label"].tolist()

train_q, test_q, train_a, test_a = train_test_split(
    questions, answers, stratify=labels, test_size=0.1, random_state=42
)
print(len(train_q), len(test_q))

In [ ]:
vocab, stoi, itos = create_vocab(train_q + train_a, min_freq=2)  # min_freq
print(f"vocab size: {len(vocab)}")

In [ ]:
MAX_LEN = 30

config = dict(
    d_model=128,
    num_layers=2,
    num_heads=4,
    d_ff=256,
    max_len=MAX_LEN + 1,
    dropout=0.3,  # dropout
)

model = ChatbotTransformer(vocab_size=len(vocab), pad_idx=stoi["<pad>"], **config).to(device)
model

In [ ]:
train_dataset = ChatbotDataset(train_q, train_a, stoi, max_len=MAX_LEN)
test_dataset = ChatbotDataset(test_q, test_a, stoi, max_len=MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

src, tgt_in, tgt_out = next(iter(train_dataloader))
src.shape, tgt_in.shape, tgt_out.shape

In [ ]:
PAD_IDX = stoi["<pad>"]

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)  # label smoothing
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)  # AdamW, weight decay

LOG_INTERVAL = 50
NUM_EPOCHS = 30

# early stopping
BEST_PATH = "best_chatbot_transformer.pt"
PATIENCE = 3
best_loss, patience_cnt = float("inf"), 0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    with tqdm(train_dataloader, desc=f"[Epoch {epoch+1}/{NUM_EPOCHS}] Train", leave=False) as train_bar:
        for batch_idx, (src, tgt_in, tgt_out) in enumerate(train_bar):
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

            logits = model(src, tgt_in)

            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            train_loss += loss.item() * src.size(0)

            if (batch_idx + 1) % LOG_INTERVAL == 0:
                train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_dataloader.dataset)

    model.eval()
    eval_loss = 0.0

    with torch.no_grad(), tqdm(test_dataloader, desc=f"[Epoch {epoch+1}/{NUM_EPOCHS}] Eval", leave=False) as eval_bar:
        for src, tgt_in, tgt_out in eval_bar:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            eval_loss += loss.item() * src.size(0)

    eval_loss /= len(test_dataloader.dataset)

    print(f"epoch {epoch+1}/{NUM_EPOCHS} | train_loss {train_loss:.4f} | eval_loss {eval_loss:.4f}")

    # early stopping
    if eval_loss < best_loss:
        best_loss, patience_cnt = eval_loss, 0
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  -> best (eval_loss {best_loss:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print("early stopping")
            break

# best model
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
print(f"best eval_loss {best_loss:.4f}")

In [ ]:
checkpoint_path = "chatbot_transformer_013.pt"  # 013

torch.save({
    "model_state_dict": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "config": config,
}, checkpoint_path)

print(f"모델 저장 완료: {checkpoint_path}")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import shutil

shutil.copy('chatbot_transformer.pt', '/content/drive/MyDrive/chatbot_transformer.pt')
print("Drive에 복사 완료!")

In [ ]:
for q in ["안녕", "오늘 너무 피곤해", "밥 먹었어?"]:
    print(f"Q: {q}")
    print(f"A: {generate_reply(model, q, stoi, itos, max_len=MAX_LEN, device=device)}")
    print()

In [ ]:
while True:
    text = input("나: ")
    if text == "exit":
        break

    reply = generate_reply(model, text, stoi, itos, max_len=MAX_LEN, device=device)
    print(f">> 챗봇: {reply}")